    # Читаем файл c идентификаторами торговых инструментов занятых в торговле
    # Загружаем Базу с символами по которым переносятся сделки
    # Cоздание и форматирование таблицы символов CRM занятых в торговле
    # Объединяем ДФ с данными из CRM и MT5 server, расчёт необходимых значений
    # Установка спреда и смещения спреда

In [1]:
import pandas as pd
import sys
import os
import time

current_dir = os.getcwd()                                               # Определяем путь к текущему файлу (где выполняется код)
parent_dir = os.path.dirname(current_dir)                               # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта {parent_dir}")
config_path = os.path.join(parent_dir, "directory_config.txt")          # Определяем путь к файлу конфигурации

directories = {}                                                        # Читаем конфигурационный файл и создаём словарь с путями
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.split("#")[0].strip()  # Убираем комментарии и пробелы
            if "=" in line:
                key, value = map(str.strip, line.split("=", 1))
                directories[key] = os.path.join(parent_dir, value.strip("'\""))     # Формируем абсолютный путь
else: print(f"❌ ERROR: Файл конфигурации '{config_path}' не найден.")

for key, path in directories.items(): print(f"📂 {key}: {path}")                    # Вывод всех загруженных директорий

directory_data_temp_files   = directories["directory_data_temp_files"]
directory_data_log_files    = directories["directory_data_log_files"]
libraries_path = os.path.join(directories["directory_libraries_path"])          # Формируем путь к libraries_py каталогу с библиотеками *.py

sys.path.append(libraries_path)                                 # sys.path — это список путей, где Python ищет модули при import module_name.
if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                            # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)

if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                       # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else: print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                   # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "save_int_list_data_log_work_file",
                "pd_set_option",                        # Вывод ДФ
                "CSVLoader",
                "move_column",
                "list_print",
                "df_to_csv",
                "file_name_with_time",
                ],                           # Загрузка ДФ из CSV

    "sed_array_lib":
        [libraries_path, 
                "np_set_printoptions"],                         # Запрос SQL и вывод данных в ДФ  

    "sql_request":
        [libraries_path, 
                "pd_read_sql"],                         # Запрос SQL и вывод данных в ДФ  
                
    "mt5_api":
        [libraries_path,
                "mt_5_manager",
                "mt5manager",
                "manager_connect_with_control",
                "manager_disconnect_with_control",
                "admin_connect_with_control",
                "admin_disconnect_with_control",
                "getting_array_trading_instruments",
                "mt5_connect_with_control",
                "mt5_disconnect_with_control",
                "symbol_array_attributes"]              
                    }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import
print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

Рабочая директория проекта c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
📂 directory_data_temp_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\working_data_files
📂 directory_data_log_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\log_data_files
📂 directory_libraries_path: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py
✅ Каталог c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py успешно добавлен в sys.path

 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['save_int_list_data_log_work_file', 'pd_set_option', 'CSVLoader', 'move_column', 'list_print', 'df_to_csv', 'file_name_with_time']
Импорт из 'sed_array_lib' успешен: ['np_set_printoptions']
Импорт из 'sql_request' успешен: ['pd_read_sql']
Импорт из 'mt5_api' успешен: ['mt_5_manager', 'mt5manager', 'manager_connect_with_control', 'manager_disconnect_with_control', 'admin_connect_with_control

In [4]:
# Формирование таблицы информации по символам CRM c ценами и спредами
# Читаем файл c идентификаторами торговых инструментов занятых в торговле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_name = 'unique_currency_id_statement.csv'                                          # Сформирован в файле [migration_trade_acc_history.ipynb]
print(f"читаем Читаем файл c идентификаторами торговых инструментов занятых в торговле сформированный из [{file_name}]")
file_path_temp = os.path.abspath(os.path.join(parent_dir, directory_data_temp_files, file_name))    # Преобразуем в абсолютный путь
print(f"Полный путь к файлу в директории с временными файлами: {file_path_temp}")
with open(file_path_temp, 'r') as file: currency_id = file.read().strip().split(',')                # Извлекаем строки и делим их по запятой
unique_currency_id_statement_list = [int(id.strip()) for id in currency_id]                                               # Преобразуем идентификаторы в целые числа
imported["list_print"](currency_id, "идентификаторы торг.инстр. занятых в истории и открытых позициях")

читаем Читаем файл c идентификаторами торговых инструментов занятых в торговле сформированный из [unique_currency_id_statement.csv]
Полный путь к файлу в директории с временными файлами: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\working_data_files\unique_currency_id_statement.csv

 [324] элементов в списке [идентификаторы торг.инстр. занятых в истории и открытых позициях] список: ['1', '2', '4', '5', '6', '8', '9', '10', '11', '12', '13', '15', '16', '17', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '30', '31', '32', '33', '35', '36', '39', '41', '42', '43', '45', '46', '48', '49', '51', '52', '54', '56', '60', '62', '63', '64', '65', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '96', '101', '102', '104', '105', '106', '108', '120', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', 

In [5]:
# Загружаем Базу с символами по которым переносятся сделки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
file_name = "crm_symbol_df.csv"
file_path = os.path.join(directory_data_temp_files, file_name)

csv_loader = imported["CSVLoader"](file_path, delimiter=',', encoding='ISO-8859-1', df_name='symbol_mapinr_df')     # вызываем класс  Создаём ДФ из CSV Файла
crm_symbol_df = csv_loader.load_data()  
#acc_set = list({int(x) for x in set(total_deal_df['account_id'])})                  # Создание множества из колонки Преобразуем значения множества в целые числа
crm_symbol_df["mapping_fc"] = crm_symbol_df["mapping"] + ".fc"

# меняем местами колонки в ДФ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
list_col_name = ["mapping", "mapping_fc"]
crm_symbol_df = imported['move_column'](crm_symbol_df, list_col_name, new_position=0, new_position_step=1)

imported["pd_set_option"]("DF с данными торговых инструментов", crm_symbol_df, 3)

[class CSVLoader]: DataFrame 'symbol_mapinr_df' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\working_data_files\crm_symbol_df.csv'.
 
 [dif] Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['mapping', 'mapping_fc'];  
 new_position = 0, new_position_step = 1
DF с данными торговых инструментов


,mapping,mapping_fc,currency_id,trading_group_id,parent_currency_id,currency_name,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_contract_size,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at
0,EURUSD,EURUSD.fc,1,1,1,EUR/USD,EURUSD,EURUSD,Euro vs US Dollar,USD,EUR,5,0.01,0.0001,100000.0,0,100000.0,0.0,0,0,0,1440,0,1440,0,1440,0,1440,0,1440,0,0,400,9,0,0,NaN,NaN,NaN,1.12559,1.12567,1,0,1,2025-05-19 11:55:26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,TIP.ETF,TIP.ETF.fc,974,1,974,iShares TIPS Bond ETF,TIP,TIP,NaN,USD,USD,3,0.01,0.0100,100.0,2,10.0,0.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,NaN,NaN,NaN,108.60000,108.60000,3,0,1,2025-05-17 03:15:50


In [6]:
# Получаем информацию пр ценам из црм <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
unique_currency_id_statement_str = ', '.join(f"'{symbol}'" for symbol in unique_currency_id_statement_list)
query_symbols = f"""SELECT currency_name, currency_id, sell_last_value, buy_last_value, symbol_digits  FROM `br-stone`.`__currency` WHERE currency_id IN ({unique_currency_id_statement_str});"""

# Cоздание и форматирование таблицы символов CRM занятых в торговле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
currency_crm_df = imported["pd_read_sql"](query_symbols)                                                    # Выполнение SQL-запроса и создание DataFrame
currency_crm_df["spread_crm"] = currency_crm_df["buy_last_value"] - currency_crm_df["sell_last_value"]          # Вычисление спреда
currency_crm_df["median_price_crm"] = (currency_crm_df["buy_last_value"] + currency_crm_df["sell_last_value"]) / 2  # Вычисление медианной цены

currency_crm_df = currency_crm_df.merge(crm_symbol_df[["currency_id", "mapping", "mapping_fc"]], on="currency_id", how="left")

imported["pd_set_option"]("DF с данными торговых инструментов", currency_crm_df, 3)

DF с данными торговых инструментов


,currency_name,currency_id,sell_last_value,buy_last_value,symbol_digits,spread_crm,median_price_crm,mapping,mapping_fc
0,EUR/USD,1,1.12598,1.12606,5,0.00008,1.12602,EURUSD,EURUSD.fc
...,...,...,...,...,...,...,...,...,...
323,iShares TIPS Bond ETF,974,108.60000,108.60000,3,0.00000,108.60000,TIP.ETF,TIP.ETF.fc


In [7]:
print("Запрос массив торговых инструментов MT5 server") #<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
symbol_array = imported["getting_array_trading_instruments"]('FC\\*') #('FOREX\\Fx Cross Rates\\AUDCAD, FOREX\\Fx Cross Rates\\AUDCHF')

Запрос массив торговых инструментов MT5 server
[def] Функция получения массива торговых инструментов [getting_array_trading_instruments]
<getting_array_trading_instruments>: symbols_list =  FC\*
MT5manager connect: True
SymbolTotal = 4757
len_symbol_array =  294
manager.Disconnect() =  True


In [8]:
# Создание ДФ массива Торг.Инстр. MT5 server по атрибутам <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
attributes = ['Symbol', 'Path', 'CurrencyProfit', 'ContractSize', 'Digits', 'Description']                          # Атрибуты для элементов массива
attributes_last_tick = ['datetime', 'bid', 'ask']                                                                   # Атрибуты для объекта TickLastRaw
symbol_update_data_df = imported['symbol_array_attributes'](symbol_array, attributes, attributes_last_tick)         # Создаём DF из массива
symbol_update_data_df["Digits"] = symbol_update_data_df["Digits"].astype(int)
symbol_update_data_df["spread_mt5"] = symbol_update_data_df["ask"] - symbol_update_data_df["bid"]                # Вычисляем спред
symbol_update_data_df["median_price_mt5"] = symbol_update_data_df.apply(lambda row: round((row["ask"] + row["bid"]) / 2, int(row["Digits"])),axis=1)
imported["pd_set_option"]("symbol_update_data_df:", symbol_update_data_df, 3) 

MT5manager connect: True
manager.SelectedAddAll() =  True
manager.SelectedTotal() =  4757
a = <MT5Manager.MTConSymbol object at 0x000001CDD116E3F0>; attr = Symbol; last_tick = <MT5Manager.MTTickShort object at 0x000001CDD99D2970>; atr_last_tick = datetime
a = <MT5Manager.MTConSymbol object at 0x000001CDD116E3F0>; attr = Symbol; last_tick = <MT5Manager.MTTickShort object at 0x000001CDD99D2970>; atr_last_tick = bid
a = <MT5Manager.MTConSymbol object at 0x000001CDD116E3F0>; attr = Symbol; last_tick = <MT5Manager.MTTickShort object at 0x000001CDD99D2970>; atr_last_tick = ask
{'Symbol': 'EURUSD.fc', 'datetime': '2025-05-19 14:25:50', 'bid': 1.12634, 'ask': 1.12677, 'Path': 'FC\\FOREX\\Fx Majors\\EURUSD.fc', 'CurrencyProfit': 'USD', 'ContractSize': 100000.0, 'Digits': 5, 'Description': 'EUR / USD'}
a = <MT5Manager.MTConSymbol object at 0x000001CDD9AABC50>; attr = Symbol; last_tick = <MT5Manager.MTTickShort object at 0x000001CDD99D2970>; atr_last_tick = datetime
a = <MT5Manager.MTConSymbol ob

,Symbol,Path,CurrencyProfit,ContractSize,Digits,Description,datetime,bid,ask,spread_mt5,median_price_mt5
0,EURUSD.fc,FC\FOREX\Fx Majors\EURUSD.fc,USD,100000.0,5,EUR / USD,2025-05-19 14:25:50,1.12634,1.12677,0.00043,1.12655
...,...,...,...,...,...,...,...,...,...,...,...
293,TIP.ETF.fc,FC\STOCKS\CFDs - Stocks US\TIP.ETF.fc,USD,10.0,2,iShares TIPS Bond,2025-05-19 07:25:12,108.79000,108.94000,0.15000,108.87000


In [ ]:
# Объединяем ДФ с данными из CRM и MT5 server, расчёт необходимых значений <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
merged_df = currency_crm_df.merge(symbol_update_data_df, left_on="mapping_fc", right_on="Symbol", how="left")
no_match_df = merged_df[merged_df["Symbol"].isna()].copy()  # Строки, по которым НЕ нашлось соответствия — правые поля будут NaN
imported["pd_set_option"]("no_match_df:", no_match_df, 10) 
currency_crm_mt5_df = merged_df[merged_df["Symbol"].notna()].copy()  # Строки, по которым нашлось соответствие (оставляем их)

currency_crm_mt5_df["spread_diff"]      = ((currency_crm_mt5_df["spread_crm"] - currency_crm_mt5_df["spread_mt5"]) * (10**currency_crm_mt5_df["Digits"])).round().astype(int)   # Вычисление разницы спредов
currency_crm_mt5_df["median_price_diff"] = ((currency_crm_mt5_df["median_price_crm"] - currency_crm_mt5_df["median_price_mt5"]) * (10**currency_crm_mt5_df["Digits"])).round().astype(int)
currency_crm_mt5_df["digits_diff"]      = currency_crm_mt5_df["symbol_digits"] - currency_crm_mt5_df["Digits"]
currency_crm_mt5_df["spread_crm_pips"]  = (currency_crm_mt5_df["spread_crm"] * (10**currency_crm_mt5_df["Digits"])).round().astype(int)

# меняем местами колонки в ДФ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
list_col_name = ["Symbol","spread_crm_pips", "spread_crm", "spread_diff", "median_price_diff"]
currency_crm_mt5_df = imported['move_column'](currency_crm_mt5_df, list_col_name, new_position=0, new_position_step=1)

unique_values_str = ", ".join(map(str, currency_crm_mt5_df["mapping_fc"].dropna().unique()))
unique_values = currency_crm_mt5_df["mapping_fc"].unique().tolist()


imported["pd_set_option"]("currency_crm_mt5_df:", currency_crm_mt5_df, 3) 

no_match_df:


,currency_name,currency_id,sell_last_value,buy_last_value,symbol_digits,spread_crm,median_price_crm,mapping,mapping_fc,Symbol,Path,CurrencyProfit,ContractSize,Digits,Description,datetime,bid,ask,spread_mt5,median_price_mt5
36,USD/RUB,48,87.9600,87.9600,4,0.00,87.9600,USDRUB,USDRUB.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,FTSE100,60,7690.2000,7693.0000,2,2.80,7691.6000,UK100_,UK100_.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,DAX,69,16805.5000,16807.5000,2,2.00,16806.5000,GER30,GER30.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,#AMERICAN_E,140,182.9900,183.2300,2,0.24,183.1100,#AMERICAN_,#AMERICAN_.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,#FACEBOOK,166,336.0300,336.3000,2,0.27,336.1650,FB,FB.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,CBOE Volatility Index,501,19.9400,19.9400,2,0.00,19.9400,VIX,VIX.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228,Vodafone Group,553,76.2740,76.2740,2,0.00,76.2740,VOD.L,VOD.L.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
232,Activision Blizzard Inc,558,94.5400,94.5400,2,0.00,94.5400,ATVI,ATVI.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278,SmileDirectClub Inc,605,0.0602,0.0602,2,0.00,0.0602,SDC,SDC.fc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


 
 [dif] Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['Symbol', 'spread_crm_pips', 'spread_crm', 'spread_diff', 'median_price_diff'];  
 new_position = 0, new_position_step = 1
currency_crm_mt5_df:


,Symbol,spread_crm_pips,spread_crm,spread_diff,median_price_diff,currency_name,currency_id,sell_last_value,buy_last_value,symbol_digits,median_price_crm,mapping,mapping_fc,Path,CurrencyProfit,ContractSize,Digits,Description,datetime,bid,ask,spread_mt5,median_price_mt5,digits_diff
0,EURUSD.fc,8,0.00008,-35,-53,EUR/USD,1,1.12598,1.12606,5,1.12602,EURUSD,EURUSD.fc,FC\FOREX\Fx Majors\EURUSD.fc,USD,100000.0,5.0,EUR / USD,2025-05-19 14:25:50,1.12634,1.12677,0.00043,1.12655,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,TIP.ETF.fc,0,0.00000,-15,-27,iShares TIPS Bond ETF,974,108.60000,108.60000,3,108.60000,TIP.ETF,TIP.ETF.fc,FC\STOCKS\CFDs - Stocks US\TIP.ETF.fc,USD,10.0,2.0,iShares TIPS Bond,2025-05-19 07:25:12,108.79000,108.94000,0.15000,108.87000,1.0


In [ ]:
# Установка спреда и смещения спреда <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
fix_spread = int(0) # Фиксированный спред, если "0" то используем спред из CRM

# Параметры установки соединения с сервером MetaTrader 5 <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
return_admin, return_manager = True, True # Определяем нужные объекты
pump_mode = 'SYMBOLS'                     # Режим работы с контролем
# Подключаемся к серверу MetaTrader 5 <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
import MT5Manager
admin, manager = imported["mt5_connect_with_control"](admin if 'admin' in globals() else None,  manager if 'manager' in globals() else None, return_admin, return_manager, pump_mode)
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
if (return_admin == True and admin is not None) or (return_manager == True and manager is not None):    # При успешном создании объектов начинаем создание новых символов 
    not_found_symbol_in_mt_df = pd.DataFrame(columns = crm_symbol_df.columns)                           # ДФ для символов не найденных на стороне МТ5

    print(f"unique_values = {unique_values}")
    print("manager.SelectedAddBatch", manager.SelectedAddBatch(unique_values))

    for index, row in currency_crm_mt5_df.iterrows():                                       # Проход по строкам DataFrame сверху вниз
        symbol_mapping = row['mapping_fc']         # Получаем имя символа из DataFrame
        symbol_request_result = manager.SymbolRequest(symbol_mapping)                           # Запрос с сервера конфигурации символа по имени.
        if symbol_request_result:
            print(f"\n ✅ Символ {symbol_mapping} найден на сервере MetaTrader 5")

            symbol_digits = symbol_request_result.Digits                            # Получение и установка количества знаков после запятой в цене символа.
            symbol_spread = symbol_request_result.Spread                            # Получение и установка размера спреда символа.
            symbol_spread_balance = symbol_request_result.SpreadBalance             # Получение и установка баланса спреда символа.
            symbol_spread_diff = symbol_request_result.SpreadDiff                   # Получение и установка разницы спреда символа.
            symbol_spreadDiff_balance = symbol_request_result.SpreadDiffBalance     # Получение и установка баланса разницы спреда.
            print(f"symbol_digits = {symbol_digits}, symbol_spread = {symbol_spread}, symbol_spread_balance = {symbol_spread_balance}, symbol_spread_diff = {symbol_spread_diff}, symbol_spreadDiff_balance = {symbol_spreadDiff_balance}")


            # Установка спреда<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
            if fix_spread < 1:
                spread_crm = row['spread_crm']
                if spread_crm < 1: symbol_request_result.Spread = int(1)
                else: symbol_request_result.Spread = int(row['spread_crm'])
            else: symbol_request_result.Spread = int(fix_spread)
            
            symbol_request_result.SpreadBalance = int(row['median_price_diff'])     # Установка смещения спреда

            symbol_update_result = admin.SymbolUpdate(symbol_request_result)        # Обновление символа на сервере MetaTrader 5


            if symbol_update_result:
                symbol_request_result = manager.SymbolRequest(symbol_request_result.Symbol)
                print(f"✅ Успешное обновление symbol_spread = {symbol_request_result.Symbol}, symbol_spread = {symbol_request_result.Spread}, symbol_spread_balance = {symbol_request_result.SpreadBalance}")

                last_tick_bid = 'bid'
                last_tick_ask = 'ask'
                last_tick_datetime = 'datetime'

                last_tick = manager.TickLast(symbol_mapping)                # tick [out]  Ссылка на структуру, описывающую котировку (MTTickShort).
                print(f"last_tick = {last_tick}, last_tick_bid = {last_tick.bid}, last_tick_ask = {last_tick.ask}, last_tick_datetime = {last_tick.datetime}")
                
                print(dir(last_tick))

                def clone_tick_short(tick):
                    new_tick = type(tick)()
                    for attr in dir(tick):
                        # Исключаем служебные и вызываемые атрибуты
                        if attr.startswith('__') or callable(getattr(tick, attr)):
                            continue
                        try:
                            setattr(new_tick, attr, getattr(tick, attr))
                        except AttributeError:
                            pass  # если атрибут только для чтения или не устанавливается
                    return new_tick
                
                last_tick_2 = last_tick#clone_tick_short(last_tick)


                print(f"row['buy_last_value']= {row['buy_last_value']}, row['sell_last_value'] = {row['sell_last_value']}")
                last_tick_2.ask = row['buy_last_value']
                last_tick_2.bid = row['sell_last_value']
                last_tick_2.last = row['buy_last_value']
                
                del last_tick
                print(f"last_tick_2 = {last_tick_2}, last_tick_2.ask = {last_tick_2.ask}, last_tick_2.bid = {last_tick_2.bid}")


                #del last_tick
                list_last_tick = [last_tick_2]

                
                
                tick_history_replace = manager.TickHistoryReplace(symbol_mapping, last_tick_2.datetime_msc-1, last_tick_2.datetime_msc+1, last_tick_2)  # tick [in]  Массив структур MTTickShort, описывающих добавляемые тики.
                print(f"tick_history_replace = {tick_history_replace}")
                del last_tick_2
                
                if admin.TickAdd(symbol_mapping, list_last_tick) == False:     # ticks [in]  Массив структур MTTickShort, описывающих добавляемые тики.
                    print(f"❌ Error admin.TickAdd: {MT5Manager.LastError()}")
                else:
                    print(f"✅ Успешное обновление admin.TickAdd")
                    last_tick_3 = manager.TickLast(symbol_mapping)                # tick [out]  Ссылка на структуру, описывающую котировку (MTTickShort).
                    print(f"last_tick = {last_tick_3}, last_tick_bid = {last_tick_3.bid}, last_tick_ask = {last_tick_3.ask}, last_tick_datetime = {last_tick_3.datetime}")

                del last_tick_3, list_last_tick


                """if manager.TickAdd(symbol_mapping, last_tick_2) == False:     # tick [in]  Ссылка на структуру MTTick, описывающую котировку.
                    print(f"❌ Error manager.TickAdd: {MT5Manager.LastError()}")
                else: print(f"✅ Успешное обновление manager.TickAdd")"""


                """last_tick.ask = row['buy_last_value']
                last_tick.bid = row['sell_last_value']
                print(f"last_tick.ask = {last_tick.ask}, last_tick.bid = {last_tick.bid}")"""

                #getattr(last_tick, attr, None)
                #print(f"symbol_mapping = {symbol_mapping}, last_tick = {last_tick}, {attr} = {getattr(last_tick, attr, None)}")
                #print(f"Error: {MT5Manager.LastError()}")




            else:
                print(f"❌ Ошибка обновления символа {symbol_mapping} на сервере MetaTrader 5")
                #not_found_symbol_in_mt_df = pd.concat([not_found_symbol_in_mt_df, pd.DataFrame([row])], ignore_index=True)

        else:
            print("❌ ERROR: символ не найден", symbol_mapping)
            not_found_symbol_in_mt_df = pd.concat([not_found_symbol_in_mt_df, pd.DataFrame([row])], ignore_index=True)

    print(f"manager.SelectedDeleteAll() = {manager.SelectedDeleteAll()}") #

    # Закрываем соединение с сервером MetaTrader 5 <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    imported["mt5_disconnect_with_control"](admin, manager, return_admin, return_manager) #
    #''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

imported["pd_set_option"]("\n ДФ  [ not_found_symbol_in_mt_df ]:", not_found_symbol_in_mt_df, 30)

Функция подключения к MT5: return_admin = True; return_manager = True
admin.Disconnect() True
Присутствовало не завешенное соединение MT5 администратор
MT5admin connect: True
Объект mt5manager: <MT5Manager.ManagerAPI object at 0x000001CDE2ECC3F0> Режим пампинга: SYMBOLS
manager.Disconnect() True
Присутствовало не завешенное соединение MT5 менеджер
подключение в режиме пампинга EnPumpModes.PUMP_MODE_SYMBOLS)
MT5manager connect: True
unique_values = ['EURUSD.fc', 'GBPUSD.fc', 'EURJPY.fc', 'BTCUSD.fc', 'NZDUSD.fc', 'EURCHF.fc', 'USDCHF.fc', 'AUDUSD.fc', 'USDCAD.fc', 'EURGBP.fc', 'ETHUSD.fc', 'EURAUD.fc', 'USDCNH.fc', 'CADJPY.fc', 'GBPJPY.fc', 'AUDNZD.fc', 'AUDCAD.fc', 'AUDCHF.fc', 'AUDJPY.fc', 'EURNZD.fc', 'CHFJPY.fc', 'EURCAD.fc', 'CADCHF.fc', 'NZDJPY.fc', 'NZDCAD.fc', 'GBPCAD.fc', 'GBPAUD.fc', 'GBPNZD.fc', 'NZDCHF.fc', 'USDDKK.fc', 'EURPLN.fc', 'EURNOK.fc', 'EURSEK.fc', 'EURZAR.fc', 'USDZAR.fc', 'EURTRY.fc', 'EURHUF.fc', 'CHFHUF.fc', 'EURRUB.fc', 'EURDKK.fc', 'EURILS.fc', 'US500.fc', 'C

,mapping,mapping_fc,currency_id,trading_group_id,parent_currency_id,currency_name,currency_key_lr,_symbol,symbol_description,symbol_profit,symbol_margin,symbol_digits,symbol_point,symbol_point_tech,symbol_multiply,symbol_calc_mode,symbol_contract_size,symbol_margin_initial,0_Open,0_Close,1_Open,1_Close,2_Open,2_Close,3_Open,3_Close,4_Open,4_Close,5_Open,5_Close,6_Open,6_Close,leverage,spread,swap_buy,swap_sell,minimum_change,commission,expiration,sell_last_value,buy_last_value,asset_id,asset_priority,asset_active,updated_at


In [ ]:
    unique_mt5_symbol = [", ".join(crm_symbol_df["mapping_fc"].dropna().astype(str).unique())]
    print(f"{type(unique_mt5_symbol)} Уникальные символы в ДФ: {unique_mt5_symbol}")

In [ ]:
import MT5Manager
file_name = "crm_symbol_df.csv"
# [ОБЯЗАТЕЛЕН] <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
file_path = os.path.join(directory_data_temp_files, file_name)

csv_loader = imported["CSVLoader"](file_path, delimiter=',', encoding='ISO-8859-1', df_name='symbol_mapinr_df')     # вызываем класс  Создаём ДФ из CSV Файла
crm_symbol_df = csv_loader.load_data()
crm_symbol_df["mapping_fc"] = crm_symbol_df["mapping"] + ".fc"          # Добавляем к символам .fc
crm_symbol_df["mt5_bid"] = None
crm_symbol_df["mt5_ask"] = None

# меняем местами колонки в ДФ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
list_col_name = ["mapping_fc","mt5_bid", "mt5_ask"]
crm_symbol_df = imported['move_column'](crm_symbol_df, list_col_name, new_position=0, new_position_step=1)
imported["pd_set_option"]("\n ДФ  [ crm_symbol_df.csv ]:", crm_symbol_df, 3)



In [ ]:
# Модуль сбора протоколирования спредов и цен торговых инструментов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
spread_log_df = pd.DataFrame()
file_name = "spread_log_df.csv"

for _ in range(500):

    unique_currency_id_statement_str = ', '.join(f"'{symbol}'" for symbol in unique_currency_id_statement_list)
    query_symbols = f"""SELECT currency_name, currency_id, sell_last_value, buy_last_value  FROM `br-stone`.`__currency` WHERE currency_id IN ({unique_currency_id_statement_str});"""

    # Cоздание и форматирование таблицы символов CRM занятых в торговле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    currency_crm_df = imported["pd_read_sql"](query_symbols)                                                    # Выполнение SQL-запроса и создание DataFrame
    currency_crm_df["spread"] = currency_crm_df["buy_last_value"] - currency_crm_df["sell_last_value"]          # Вычисление спреда
    imported["pd_set_option"]("DF с данными торговых инструментов", currency_crm_df, 3)

    df = currency_crm_df.copy()                                                                                           # Создаем копию DataFrame для обработки
    df["combined_name"] = df["currency_name"] + " (" + df["currency_id"].astype(str) + ")"
    spread_row = df.set_index("combined_name")["spread"].T.to_frame().T
    #spread_row["timestamp"] = datetime.now()
    imported["pd_set_option"]("Добавляемая строка", spread_row, 3)
    spread_log_df = pd.concat([spread_log_df, spread_row], ignore_index=True)

    
    imported["df_to_csv"](spread_log_df,  imported["file_name_with_time"](file_name, directory_data_temp_files, time_in_name = False))
    time.sleep(10)

In [ ]:
file_name = "spread_log_df.csv"
file_path = os.path.join(directory_data_temp_files, file_name)                              

csv_loader = imported["CSVLoader"](file_path, delimiter=',', encoding='ISO-8859-1', df_name='symbol_mapinr_df')     # вызываем класс  Создаём ДФ из CSV Файла
spread_log_df = csv_loader.load_data()  

In [ ]:
imported["pd_set_option"]("DF с данными торговых инструментов", spread_log_df, 30)
stats = pd.DataFrame({
    "min":      spread_log_df.min(),
    "mean":     spread_log_df.mean(),
    "median":   spread_log_df.median(),
    "max":  spread_log_df.max()
})
imported["pd_set_option"]("Статистика по спреду", stats, 30)
                          
filtered_stats = stats[(stats["min"] != stats["mean"]) | (stats["min"] != stats["max"]) | (stats["mean"] != stats["max"])]
print(filtered_stats)


In [ ]:
stats = pd.DataFrame({
    "min": spread_log_df.min(),
    "mean": spread_log_df.mean(),
    "max": spread_log_df.max()
})


In [ ]:

    unique_mt5_symbol = ["EURUSD.fc"] # Удалить после отладки
    print(f"{type(unique_mt5_symbol)} EURUSD, GBPUSD, USDJPY: {unique_mt5_symbol}")

    print("manager.SelectedAddBatch", manager.SelectedAddBatch(unique_mt5_symbol))                              # Добавляем символы в список для запроса

    symbol_selected_total = manager.SelectedTotal()                               # Получаем количество символов в списке для запроса
    print(f"Количество символов в списке для запроса: {symbol_selected_total}, {manager.SelectedNext(symbol_selected_total-1)}")



